# Recommandation d'artistes avec K-Means

Ce notebook montre uniquement l'utilisation du pipeline défini dans `src/recommend.py`. Chaque résultat est rattaché à la version exacte du dataset nettoyé.

In [30]:
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "8")

import importlib
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "src").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Impossible de trouver la racine MusicRecoSystem")

sys.path.insert(0, str(PROJECT_ROOT))

import src.recommend as recommend_module
importlib.reload(recommend_module)
from src.recommend import select_and_fit_best_kmeans

## 1. Charger une version précise des données

La colonne `dataset_version` permet de vérifier que deux expériences utilisent exactement le même nettoyage.

In [31]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "listens.parquet"
listens = pd.read_parquet(DATA_PATH)

dataset_versions = listens["dataset_version"].unique()
if len(dataset_versions) != 1:
    raise ValueError("Le fichier mélange plusieurs versions de données")

pd.Series({
    "dataset_version": dataset_versions[0],
    "interactions": len(listens),
    "users": listens["user"].nunique(),
    "artist_identities": listens["artist_uid"].nunique(),
    "mbid_coverage": listens["artist_mbid"].notna().mean(),
})

dataset_version      clean-v1.0.0-19b966482277
interactions                             10064
users                                       82
artist_identities                         2640
mbid_coverage                         0.909976
dtype: object

## 2. Trouver le meilleur k et entraîner la solution finale

Cette cellule réalise la séparation chronologique 80/10/10, sélectionne `k` sur la validation, réentraîne sur train + validation et évalue une seule fois sur le test.

In [32]:
import importlib
import src.recommend as recommend_module
recommend_module = importlib.reload(recommend_module)

experiment = recommend_module.select_and_fit_best_kmeans(
    listens,
    k_values=[2, 3, 4, 5, 6, 8, 10, 12, 15, 20],
    top_n=10,
    random_state=42,
    verbose=True,
)

display(experiment.tuning_results)

metrics = pd.DataFrame({
    "validation": experiment.validation_metrics,
    "test_final": experiment.test_metrics,
})

experiment_version = getattr(
    experiment,
    "dataset_version",
    listens["dataset_version"].iloc[0],
)
print(f"Dataset : {experiment_version}")
print(f"Meilleur k : {experiment.best_k}")
display(metrics)

Dataset clean-v1.0.0-19b966482277 — train=8038, validation=1020, test=1006
K-Means 1/10 — entraînement avec k=2
  Recall@10=0.1264 | NDCG@10=0.1157
K-Means 2/10 — entraînement avec k=3
  Recall@10=0.1494 | NDCG@10=0.1194
K-Means 3/10 — entraînement avec k=4
  Recall@10=0.1530 | NDCG@10=0.1196
K-Means 4/10 — entraînement avec k=5
  Recall@10=0.1460 | NDCG@10=0.1112
K-Means 5/10 — entraînement avec k=6
  Recall@10=0.1212 | NDCG@10=0.0954
K-Means 6/10 — entraînement avec k=8
  Recall@10=0.1036 | NDCG@10=0.0785
K-Means 7/10 — entraînement avec k=10
  Recall@10=0.1082 | NDCG@10=0.0910
K-Means 8/10 — entraînement avec k=12
  Recall@10=0.1376 | NDCG@10=0.1054
K-Means 9/10 — entraînement avec k=15
  Recall@10=0.0885 | NDCG@10=0.0711
K-Means 10/10 — entraînement avec k=20
  Recall@10=0.1239 | NDCG@10=0.1021
Meilleur k=4 — modèle final entraîné et évalué sur le test


,k,recall@10,ndcg@10,n_evaluable_users
0,2,0.126410,0.115660,67
1,3,0.149380,0.119406,67
2,4,0.153021,0.119634,67
3,5,0.146026,0.111187,67
4,6,0.121180,0.095396,67
5,8,0.103625,0.078459,67
6,10,0.108204,0.091011,67
7,12,0.137586,0.105420,67
8,15,0.088457,0.071059,67
9,20,0.123905,0.102069,67


Dataset : clean-v1.0.0-19b966482277
Meilleur k : 4


,validation,test_final
recall@10,0.153021,0.131307
ndcg@10,0.119634,0.107454
n_evaluable_users,67.000000,66.000000


## 3. Inspecter le modèle final

In [33]:
print("Tailles des parties :", experiment.split_sizes)
print("Utilisateurs par cluster :")
experiment.recommender.cluster_sizes()

Tailles des parties : {'train': 8038, 'validation': 1020, 'test': 1006}
Utilisateurs par cluster :


cluster
0    16
1    24
2     6
3    36
Name: count, dtype: int64

## 4. Obtenir 10 recommandations

Les artistes déjà écoutés sont automatiquement exclus.

In [34]:
USER = "BKILPATRICK11"

experiment.recommender.recommend(USER, top_n=10)

,rank,artist_uid,artist
0,1,mbid:e520459c-dff4-491d-a6e4-c97be35e0044,Frank Ocean
1,2,mbid:9fff2f8a-21e6-47de-a2b8-7f449929d43f,Drake
2,3,mbid:b46e298d-d3a5-4695-a03e-63e8c523b23e,Daniel Caesar
3,4,mbid:e4a51f17-a57b-47b1-b37b-f552d0f8e9e6,Travis Scott
4,5,mbid:272989c8-5535-492d-a25c-9f58803e027f,SZA
5,6,mbid:7fb57fba-a6ef-44c2-abab-2fa3bdee607e,Childish Gambino
6,7,mbid:6c2a79b8-2b5c-4c2d-b495-6a446811bf9d,Malcolm Todd
7,8,mbid:73e5e69d-3554-40d8-8516-00cb38737a1c,Rihanna
8,9,mbid:7bd66c18-cb33-49aa-9279-140acedbdb49,Ty Dolla $ign
9,10,mbid:201412c6-7ced-472b-87a9-9259ec3c07d3,Brent Faiyaz
